## Structured Output in LangChain

Structured output means forcing the LLM to return data in a **specific format** (like a dictionary or JSON) instead of free text.  
LangChain provides multiple ways to achieve this, depending on how strict you want the schema enforcement to be.

---

### Options for Structured Output

1. **TypedDict (Python typing)**  
   - Define a schema using Python’s `TypedDict`.  
   - Lightweight, easy to use, but doesn’t enforce strict validation.  
   - Best for quick demos and teaching structured outputs.

2. **Pydantic Models**  
   - Define a schema using `pydantic.BaseModel`.  
   - Enforces **strict type validation** (e.g., ensures integers are integers).  
   - Best for production systems where data integrity matters.

3. **JSON Schema**  
   - Language-agnostic schema definition.  
   - Useful when backend and frontend are in different languages (Python ↔ React/Flutter).  
   - Ensures both sides share the same schema definition.

---

### Comparison

| Method        | Strictness | Best Use Case | Limitation |
|---------------|------------|---------------|------------|
| **TypedDict** | Low        | Quick demos, teaching | No runtime validation |
| **Pydantic**  | High       | Production apps needing type safety | More setup required |
| **JSON Schema** | Medium–High | Cross-language projects | Requires schema definition separately |

---

### Takeaway
- Use **TypedDict** when you want simplicity and speed.  
- Use **Pydantic** when you need strict validation and error handling.  
- Use **JSON Schema** when working across multiple programming languages.  


In [20]:
# TypedDict Example with Groq
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from typing import TypedDict

# Load environment variables (make sure GROQ_API_KEY is in your .env file)
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Define a schema using TypedDict
class MatchSummary(TypedDict):
    team_won: str
    team_lost: str
    winning_margin: str
    venue: str
    top_batter: str
    top_batter_runs: str
    key_moment: str

# Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant", 
api_key=groq_key)

# Attach schema for structured output
ipl_bot = llm.with_structured_output(MatchSummary)

# Invoke with a fictional IPL final description
response = ipl_bot.invoke(
    "Describe a fictional IPL 2026 final match between Mumbai Indians and RCB."
)

# Print structured results
print("Winner       :", response["team_won"])
print("Venue        :", response["venue"])
print("Top batter   :", response["top_batter"], "—", response["top_batter_runs"], "runs")
print("Key moment   :", response["key_moment"])

Winner       : Mumbai Indians
Venue        : Wankhede Stadium
Top batter   : Suryakumar Yadav — 85 runs
Key moment   : Surya Yadav's unbeaten 85 helped MI to chase down the target of 190 runs


#### Sometime you may get below error: 

## ⚠️ BadRequestError (400)

- **Error Type:** `invalid_request_error`
- **Message:** *"Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details."*
- **Cause:** Schema validation failed when the LLM tried to return structured output.
- **Failed Generation:**  
  ```python
  dict(
      team_won="Mumbai Indians",
      team_lost="RCB",
      winning_margin="6 wickets",
      venue="Wankhede Stadium",
      top_batter="Rohit Sharma",
      top_batter_runs=82,   # Returned as int
      key_moment="Kieron Pollard's 4 sixes in the final over"
  )


  #### Root Issue
- `top_batter_runs` was defined as `int` in `TypedDict`, but the LLM often outputs numbers as strings.

#### Fix
- Change schema field to `str` or use **Pydantic** for automatic type coercion.

#### Best Practice
- Define numeric fields as `str` in `TypedDict`.
- Convert to `int` in Python after receiving the response.
- Use **Pydantic models** if you want automatic conversion and validation.

---------



# Pydantic Overview

- **What is Pydantic?**
  - A Python library for **data validation and parsing** using type hints.
  - Ensures that data matches the expected schema at runtime.

- **How it works:**
  - Define a schema using `BaseModel`.
  - Pydantic automatically validates types and can **coerce strings into numbers**.
  - If the data doesn’t match, it raises a clear error.

- **Why use Pydantic with LLMs?**
  - LLMs often output numbers as strings (e.g., `"82"` instead of `82`).
  - Pydantic can **auto-convert** `"82"` → `82` without breaking.
  - Provides stronger guarantees than `TypedDict`.

- **Best Practice:**
  - Use `TypedDict` for simple demos.
  - Use `Pydantic` for production apps where **data integrity matters**.


In [ ]:
# Pydantic Example with Groq
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from pydantic import BaseModel

# Load environment variables (make sure GROQ_API_KEY is in your .env file)
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Define schema using Pydantic
class MatchSummary(BaseModel):
    team_won: str
    team_lost: str
    winning_margin: str
    venue: str
    top_batter: str
    top_batter_runs: int   # ✅ Pydantic will auto-convert "82" → 82. If throws error replace int with str.
    key_moment: str

# Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Attach schema for structured output
ipl_bot = llm.with_structured_output(MatchSummary)

# Invoke with a fictional IPL final description
response = ipl_bot.invoke(
    "Describe a fictional IPL 2026 final match between Mumbai Indians and RCB."
)

# Print structured results
print("Winner       :", response.team_won)
print("Venue        :", response.venue)
print("Top batter   :", response.top_batter, "—", response.top_batter_runs, "runs")
print("Key moment   :", response.key_moment)

BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool MatchSummary did not match schema: errors: [`/top_batter_runs`: expected integer, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=MatchSummary> {"team_lost": "RCB", "team_won": "Mumbai Indians", "winning_margin": "6 wickets", "key_moment": "Kieron Pollard\'s 50 off 20 balls", "top_batter": "Suryakumar Yadav", "top_batter_runs": "73", "venue": "DY Patil Stadium"}</function>'}}

----------

## Structured Output with OpenAI

- **OpenAI API** supports structured outputs with Pydantic models.
- Unlike Groq, OpenAI often **auto-coerces types** (e.g., `"128"` → `128`).
- You can define a schema using `BaseModel` and attach it with `with_structured_output`.
- This ensures the LLM response matches your schema, reducing manual parsing.


In [ ]:
# Pydantic Structured Output with OpenAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

# Paste your API key directly here
openai_key = "sk-XXXXXXXXXXX"

# Define schema using Pydantic
class MatchSummary(BaseModel):
    team_won: str
    team_lost: str
    winning_margin: str
    venue: str
    top_batter: str
    top_batter_runs: int   # OpenAI will coerce "128" → 128
    key_moment: str

# Initialize OpenAI LLM
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_key)

# Attach schema for structured output
ipl_bot = llm.with_structured_output(MatchSummary)

# Invoke with a fictional IPL final description
response = ipl_bot.invoke(
    "Describe a fictional IPL 2026 final match between Mumbai Indians and RCB."
)

# Print structured results
print("Winner       :", response.team_won)
print("Venue        :", response.venue)
print("Top batter   :", response.top_batter, "—", response.top_batter_runs, "runs")
print("Key moment   :", response.key_moment)


Winner       : Mumbai Indians
Venue        : Wankhede Stadium, Mumbai
Top batter   : Sky Sharma — 89 runs
Key moment   : Sky Sharma's stunning 6 off the last ball of the 18th over to bring up his half-century changed the momentum in favor of Mumbai.


#### Why It Worked with OpenAI API

- **Closed‑source models (like OpenAI GPT)** have **native support for structured outputs**.
- They are designed to **follow schema instructions more reliably** than open‑source models.
- OpenAI automatically **coerces types** when possible:
  - Example: `"128"` (string) → `128` (integer).
- This prevents the strict validation errors you saw with Groq, which requires exact type matches.
- **Groq API** enforces stricter tool call validation:
  - If schema says `int`, the model must return an integer, not a string.
  - Any mismatch leads to `BadRequestError`.
- **OpenAI API** is more forgiving:
  - It interprets the schema and adjusts the output to fit.
  - This makes it smoother for production use where LLMs often output numbers as strings.
- **Takeaway:**  
  - Groq = strict, exact schema enforcement.  
  - OpenAI = lenient, auto‑coercion for structured outputs.  

---------

## JSON Schema Overview

- **What is JSON Schema?**
  - A dictionary-based format to define the structure of JSON data.
  - Language-agnostic: works across Python, JavaScript, Flutter, or any other environment.
  - Defines keys, types, and required fields.

- **Why use it?**
  - Ensures consistent data shape across backend and frontend.
  - Useful when multiple systems (e.g., Python backend + React frontend) need to share the same schema definition.

- **Best Situation to Use:**
  - Projects where the same data structure must be enforced across different programming languages.
  - Example: APIs, cross-platform apps, or microservices exchanging structured data.

In [16]:
# Simplest JSON Schema Example with Groq
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

# Load environment variables (make sure GROQ_API_KEY is in your .env file)
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Define schema as a JSON Schema dictionary
match_schema = {
    "title": "MatchSummary",
    "type": "object",
    "properties": {
        "team_won": {"type": "string"},
        "team_lost": {"type": "string"},
        "winning_margin": {"type": "string"},
        "venue": {"type": "string"},
        "top_batter": {"type": "string"},
        "top_batter_runs": {"type": "string"},  # keep as string to avoid strict int errors
        "key_moment": {"type": "string"}
    },
    "required": ["team_won", "team_lost", "winning_margin", "venue", "top_batter", "top_batter_runs", "key_moment"]
}

# Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Attach schema for structured output
ipl_bot = llm.with_structured_output(match_schema)

# Invoke with a fictional IPL final description
response = ipl_bot.invoke(
    "Describe a fictional IPL 2026 final match between Mumbai Indians and RCB."
)

# Print structured results
print("Winner       :", response["team_won"])
print("Venue        :", response["venue"])
print("Top batter   :", response["top_batter"], "—", response["top_batter_runs"], "runs")
print("Key moment   :", response["key_moment"])

Winner       : Mumbai Indians
Venue        : Wankhede Stadium
Top batter   : Suryakumar Yadav — 120 runs
Key moment   : Suryakumar Yadav's 120 off 60 balls
